# PyTorch-Modelle „architektonisch“
wie branch1 / branch2 definiert sind

was nn.ModuleList(layers) wirklich enthält (inkl. Autograd & Parameter-Tracking)

## 1. Besipiel 

In [4]:
# definition of 2 branches in a class

import torch
import torch.nn as nn

class FancyNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()

        # Branch 1
        self.branch1 = nn.Sequential(
            nn.Linear(in_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )

        # Branch 2
        self.branch2 = nn.Sequential(
            nn.Linear(in_features, 32),
            nn.Tanh(),
            nn.Linear(32, 16)
        )
# forward for two branches
    def forward(self, x):
        x1 = self.branch1(x)   # → Sequential.forward()
        x2 = self.branch2(x)   # → Sequential.forward()
        return x1 + x2         # Residual / Merge



        

In [5]:
torch.manual_seed(0)

x = torch.randn(4, 10)
model = FancyNet(in_features=10)

y = model(x)
print("Output shape:", y.shape)
print(y)

Output shape: torch.Size([4, 16])
tensor([[-0.3990,  0.2235, -0.3671,  1.1406, -0.8533, -0.0301, -0.3376, -0.0427,
         -0.3800, -0.1037,  0.3192, -0.1572, -0.3047,  0.1445,  0.3340,  0.4933],
        [-0.1991,  0.0396, -0.0848,  0.5874, -0.2220, -0.2729,  0.0451,  0.1375,
          0.4954, -0.1717, -0.3778, -0.0404, -0.0683,  0.0874,  0.4672,  0.5837],
        [ 0.2354, -0.5587,  0.0584, -0.3257, -0.1920,  0.1093, -0.3660, -0.2630,
          0.0300,  0.3250, -0.4658, -0.7656, -0.4328,  0.2114,  0.2530,  0.1805],
        [ 0.3378, -0.5772, -0.1242, -0.2112, -0.0690,  0.5043, -0.1239, -0.3851,
          0.1147, -0.1258, -0.2913, -0.1004,  0.6159,  0.1474,  0.7516,  0.4478]],
       grad_fn=<AddBackward0>)


## 2 Beispiel Dasselbe Beispiel – aber ohne nn.Sequential
Jetzt bauen wir die Branches mit deinem MySequential

Was ist nn.ModuleList(layers) wirklich?

nn.ModuleList(): layers is a ModuleList

In [7]:
class MySequential(nn.Module):
    def __init__(self, *layers):
        super().__init__()
        self.layers = nn.ModuleList(layers)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

        

### FancyNet mit MySequential 
**self.branch1 = MySequential**(
            nn.Linear(in_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )

In [8]:
#FancyNet mit MySequential
# funktional identisch zu nn.Sequential

class FancyNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()

        self.branch1 = MySequential(
            nn.Linear(in_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )

        self.branch2 = MySequential(
            nn.Linear(in_features, 32),
            nn.Tanh(),
            nn.Linear(32, 16)
        )

    def forward(self, x):
        return self.branch1(x) + self.branch2(x)